In [ ]:
# 1. Load the Thermal Logic Class (ZoneModel)
%run "Multi_zone_model_Class_Case_Study.ipynb"

# 2. Import required libraries
import pandas as pd
import numpy as np

# =================================================================
# SECTION 1: DATA LOADING & GLOBAL CONSTANTS
# =================================================================

# --- 168-HOUR SETBACK PROFILE GENERATOR ---
# Weekdays: 
# 00:00 - 06:00: Stable Setback (18 / 26)
# 06:00 - 09:00: Smooth Ramp (18.5 -> 19.0 -> 19.5)
# 09:00 - 00:00: Comfort (20 / 24)

h_day = [18.0]*6 + [18.5, 19.0, 19.5] + [20.0]*15  
c_day = [26.0]*6 + [25.5, 25.0, 24.5] + [24.0]*15  

# Weekend: Constant Stable Setback (18 / 26)
h_wend = [18.0]*24
c_wend = [26.0]*24

# 5 active days, weekend off
heating_profile_5 = (h_day * 5) + (h_wend * 2)
cooling_profile_5 = (c_day * 5) + (c_wend * 2)

# 7 active days, no weekend setback
heating_profile_7 = (h_day * 7) 
cooling_profile_7 = (c_day * 7) 

# Define ONLY the base inputs that are truly universal (Building-wide) 
# Any zone specific inputs (like floor area, facade areas, etc.) will be defined in the individual zone dictionaries below and override these base inputs.
base_inputs = {
    # --- Geometry & Location ---
    "t_ground": 15.0,

    # --- Setback settings ---
    "heating_setpoint_profile": heating_profile_7,
    "cooling_setpoint_profile": cooling_profile_7,
    "vent_comfort_limit": 23.5, # Absolute minimum temp to allow window venting
    "heating_setpoint": 20.0,
    "cooling_setpoint": 24.0,

    # --- Thermal Properties (Shared Envelope Defaults) ---
    "rc_roof": 7.84,
    "rc_ground_floor": 4.7,
    "u_value_windows": 1.84,
    "alfai": 7.5,
    "alfao": 15.0,
    "thermal_mass_factor": 450000, # Or override it in the individual zones if needed

    # --- Solar Properties ---
    "solar_absorption_coefficient": 0.5,
    "solar_heat_coefficient_shading": 0.20,
    "solar_heat_coefficient_glazing": 0.35,

    # --- Ventilation & Air Properties ---
    "air_density": 1.2,
    "air_heat_capacity": 1003.0,
    "vent_flow_per_person": 36.0,
    "infiltration_ach": 0.2,
    "natural_vent_rate": 3.0,

    # --- Internal Gains ---
    "heat_per_person": 65.0,
    "appliances_w_m2": 6.0,
    "lighting_w_m2": 6.0,

    # --- HVAC System Constants ---
    "heating_power_max": 1000000.0,     # Importance: This caps the heating capacity to prevent mathematical 'infinite power' spikes that occur when the building's thermal coupling (H_total) approaches zero.
    "cooling_power_max": -1000000.0,    # Importance: This caps the cooling capacity to prevent mathematical 'infinite power' spikes that occur when the building's thermal coupling (H_total) approaches zero.
    "system_pressure_drop": 200.0,
    "efficiency_fan_and_motor": 0.6,
    "eta": 0.0, # heat recovery factor

    # --- NEW: Floor heating added ---
    "u_floor_heating": 11.0,                # W/m2K, a typical value for underfloor heating systems
    "floor_heating_area": 200,   # Actual area with heating pipes
    "t_floor_water_heating": 35.0,          # °C, supply temperature for underfloor heating
    "t_floor_water_cooling": 18.0,          # °C, supply temperature for underfloor cooling
    "t_floor_limit_heating": 24, # The floor stops heating when Atrium hits 22.5°C
    "t_floor_limit_cooling": 18.0, # The floor stops cooling when Atrium hits 20.0°C

    # Default Schedule fallback
    "system_profile": [1.0]*168
}

# =================================================================
# SECTION 2: SHARED WALL CONNECTIONS (ADJACENCIES)
# =================================================================

# Define physical properties of shared internal partitions
# Area [m2], R_value [m2K/W]
internal_wall_connections = [
    {"zones": ("Zone 1", "Zone 2"), "area": 505.0, "R": 0.33},  # Lower R-value: to couple the threate hall to the atrium
    {"zones": ("Zone 1", "Zone 4"), "area": 122.0, "R": 0.60},  # Higher R-value: to decouple the theatre hall from the offices/changing rooms
    {"zones": ("Zone 1", "Zone 6"), "area": 35.0, "R": 1.00},   # Higher R-value for zones connected to zone 6: technical spaces with no active HVAC
    {"zones": ("Zone 3", "Zone 2"), "area": 355.0, "R": 0.33},  # Lower R-value: to couple repitition spaces to the atrium
    {"zones": ("Zone 3", "Zone 4"), "area": 295.0, "R": 0.33}, 
    {"zones": ("Zone 3", "Zone 5"), "area": 525.0, "R": 0.50},  # Higher R-value for zones connected to zone5: workshops have often lower setpoints because of labor being done
    {"zones": ("Zone 3", "Zone 6"), "area": 177.0, "R": 1.00}, 
    {"zones": ("Zone 4", "Zone 2"), "area": 208.0, "R": 0.50},
    {"zones": ("Zone 4", "Zone 5"), "area": 20.0, "R": 0.50},  
    {"zones": ("Zone 4", "Zone 6"), "area": 218.0, "R": 1.00}, 
    {"zones": ("Zone 5", "Zone 6"), "area": 266.0, "R": 1.00}, 
    {"zones": ("Zone 6", "Zone 2"), "area": 359.0, "R": 1.00}, 
]

# AUTOMATION: Convert physical inputs into UA values for the simulation
adjacencies = []
for connection in internal_wall_connections:
    ua_calculated = connection["area"] / connection["R"]
    adjacencies.append({
        "zones": connection["zones"], 
        "UA": ua_calculated
    })

# =================================================================
# SECTION 3: ZONE DEFINITIONS
# =================================================================

zone_definitions = []

# --- ZONE 1: Theatre hall ---------------------------------------------------------------------------------------------------------------------------------------
z1 = base_inputs.copy()
z1.update({
    "floor_area": 455.0,
    "area_roof": 455.0,
    "area_ground": 455.0,
    "room_volume": 4077.0,

    "rc_facade": 4.86,
    "thermal_mass_factor": 450000,

    "total_facade_areas": {'N': 255.0, 'NE': 0.0, 'E': 0.0, 'SE': 0.0, 'S': 0.0, 'SW': 0.0, 'W': 0.0, 'NW': 0.0},
    "window_percentages": {'N': 0.00, 'NE': 0.00, 'E': 0.0, 'SE': 0.0, 'S': 0.0, 'SW': 0.0, 'W': 0.0, 'NW': 0.0},
    "glazing_percentages": {'N': 0.00, 'NE': 0.00, 'E': 0.00, 'SE': 0.00, 'S': 0.00, 'SW': 0.00, 'W': 0.00, 'NW': 0.00},
    
    "max_people_per_m2": 0.55,  # based on 250 people
    "appliances_w_m2": 12.0,
    "lighting_w_m2": 15.0,
    
    "heating_setpoint_profile": heating_profile_5,
    "cooling_setpoint_profile": cooling_profile_5,

    "floor_heating_activated": False,       # True or False, whether to include floor heating in the calculations
    "vent_flow_per_person": 36.0,
    "occ_profile": [
    # Monday
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
    # Tuesday
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
    # Wednesday event
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.40, 0.40, 0.40, 0.40, 1.00, 1.00, 1.00, 1.00, 0.40,
    # Thursday
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
    # Friday event
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.40, 0.40, 0.40, 0.40, 1.00, 1.00, 1.00, 1.00, 0.40,
    # Saturday event
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.40, 0.40, 0.40, 0.40, 1.00, 1.00, 1.00, 1.00, 0.40,
    # Sunday event
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.40, 0.40, 0.40, 0.40, 1.00, 1.00, 1.00, 1.00, 0.40
    ],
    "equip_profile": [
    # Monday
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
    # Tuesday
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
    # Wednesday event
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.40, 0.40, 0.40, 0.40, 1.00, 1.00, 1.00, 1.00, 0.40,
    # Thursday
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
    # Friday event
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.40, 0.40, 0.40, 0.40, 1.00, 1.00, 1.00, 1.00, 0.40,
    # Saturday event
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.40, 0.40, 0.40, 0.40, 1.00, 1.00, 1.00, 1.00, 0.40,
    # Sunday event
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.40, 0.40, 0.40, 0.40, 1.00, 1.00, 1.00, 1.00, 0.40
    ],
    "vent_profile": [
    # Monday: 0.10 represents nighttime infiltration/leakage
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Tuesday
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Wednesday event
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Thursday
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Friday event
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Saturday event
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Sunday event
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    ],
})
zone_definitions.append(("Zone 1", z1))

# --- ZONE 2: Atrium/foyer ---------------------------------------------------------------------------------------------------------------------------------------
z2 = base_inputs.copy()
z2.update({
    "floor_area": 565.0,
    "area_roof": 565.0,
    "area_ground": 565.0,
    "room_volume": 7370.0,

    "rc_roof": 13.66,   # Different Rc-value from rest of the roof structure
    "rc_facade": 4.7,   # or 8.57?
    "thermal_mass_factor": 450000,

    "total_facade_areas": {'N': 0.0, 'NE': 80.0, 'E': 0.0, 'SE': 205.0, 'S': 0.0, 'SW': 315.0, 'W': 0.0, 'NW': 230.0},
    "window_percentages": {'N': 0.0, 'NE': 0.0, 'E': 0.00, 'SE': 0.293, 'S': 0.0, 'SW': 0.143, 'W': 0.0, 'NW': 0.187},
    "glazing_percentages": {'N': 0.00, 'NE': 0.00, 'E': 0.00, 'SE': 0.80, 'S': 0.00, 'SW': 0.80, 'W': 0.00, 'NW': 0.80},
    
    "max_people_per_m2": 0.28,  # based on 160 people
    "appliances_w_m2": 10.0,
    "lighting_w_m2": 6.0,

    "floor_heating_activated": False,       # True or False, whether to include floor heating in the calculations
    "floor_profile": [
    # Monday (Example: ON from 02:00 to 09:00)
    0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
    # Tuesday
    0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
     # Wednesday
    0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
    # Thursday
    0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
    # Friday
    0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
    # Saturday 
    0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
    # Sunday
    0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
    ],
    "vent_flow_per_person": 36.0,
    "heating_setpoint_profile": [16.0] * 168,
    "cooling_setpoint_profile": [28.0] * 168,
    "vent_cooling_setpoint": 27.0,
    "occ_profile": [
    # Monday 
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30,
    # Tuesday 
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30,
    # Wednesday event
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.50, 0.50, 0.50, 0.50, 0.30, 0.30, 0.30, 0.30, 0.50,
    # Thursday 
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30,
    # Friday event
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.50, 0.50, 0.50, 0.50, 0.30, 0.30, 0.30, 0.30, 0.30,
    # Saturday event
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.50, 0.50, 0.50, 0.50, 0.30, 0.30, 0.30, 0.30, 0.50,
    # Sunday event
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.50, 0.50, 0.50, 0.50, 0.30, 0.30, 0.30, 0.30, 0.50
    ],
    "equip_profile": [
    # Monday 
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30,
    # Tuesday 
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30,
    # Wednesday event
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.50, 0.50, 0.50, 0.50, 0.30, 0.30, 0.30, 0.30, 0.50,
    # Thursday 
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30,
    # Friday event
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.50, 0.50, 0.50, 0.50, 0.30, 0.30, 0.30, 0.30, 0.30,
    # Saturday event
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.50, 0.50, 0.50, 0.50, 0.30, 0.30, 0.30, 0.30, 0.50,
    # Sunday event
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.50, 0.50, 0.50, 0.50, 0.30, 0.30, 0.30, 0.30, 0.50
    ],
    "vent_profile": [
    # Monday
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Tuesday
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Wednesday event
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Thursday
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Friday event
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Saturday event
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Sunday event
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    ],
})
zone_definitions.append(("Zone 2", z2))

# --- ZONE 3: Repetition cluster ---------------------------------------------------------------------------------------------------------------------------------------
z3 = base_inputs.copy()
z3.update({
    "floor_area": 685.0,
    "area_roof": 160.0,
    "area_ground": 525.0,
    "room_volume": 4315.0, 


    "rc_facade": 4.86, 
    "thermal_mass_factor": 450000,

    "total_facade_areas": {'N': 0.0, 'NE': 93.0, 'E': 0.0, 'SE': 265.0, 'S': 0.0, 'SW': 0.0, 'W': 0.0, 'NW': 64.0},
    "window_percentages": {'N': 0.0, 'NE': 0.032, 'E': 0.0, 'SE': 0.0, 'S': 0.0, 'SW': 0.0, 'W': 0.0, 'NW': 0.0},
    "glazing_percentages": {'N': 0.00, 'NE': 0.80, 'E': 0.00, 'SE': 0.00, 'S': 0.00, 'SW': 0.00, 'W': 0.00, 'NW': 0.00},
    
    "max_people_per_m2": 0.26,  # based on 110 people
    "appliances_w_m2": 10.0,
    "lighting_w_m2": 12.0,
    
    "floor_heating_activated": False,       # True or False, whether to include floor heating in the calculations
    "vent_flow_per_person": 25.2,
    "occ_profile": [
    # Monday
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.40, 0.40, 0.40, 0.40, 0.40, 0.40, 0.40, 1.00, 1.00, 1.00, 1.00, 0.00,
    # Tuesday 
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.40, 0.40, 0.40, 0.40, 0.40, 0.40, 0.40, 1.00, 1.00, 1.00, 1.00, 0.00,
    # Wednesday 
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.40, 0.40, 0.40, 0.40, 0.40, 0.40, 0.40, 1.00, 1.00, 1.00, 1.00, 0.00,
    # Thursday 
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.40, 0.40, 0.40, 0.40, 0.40, 0.40, 0.40, 1.00, 1.00, 1.00, 1.00, 0.00,
    # Friday
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.40, 0.40, 0.40, 0.40, 0.40, 0.40, 0.40, 1.00, 1.00, 1.00, 1.00, 0.00,
    # Saturday
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.40, 0.40, 0.40, 0.40, 0.40, 0.40, 0.40, 1.00, 1.00, 1.00, 1.00, 0.00,
    # Sunday
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.40, 0.40, 0.40, 0.40, 0.40, 0.40, 0.40, 1.00, 1.00, 1.00, 1.00, 0.00
    ],
    "equip_profile": [
    # Monday
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.40, 0.40, 0.40, 0.40, 0.40, 0.40, 0.40, 1.00, 1.00, 1.00, 1.00, 0.00,
    # Tuesday 
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.40, 0.40, 0.40, 0.40, 0.40, 0.40, 0.40, 1.00, 1.00, 1.00, 1.00, 0.00,
    # Wednesday 
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.40, 0.40, 0.40, 0.40, 0.40, 0.40, 0.40, 1.00, 1.00, 1.00, 1.00, 0.00,
    # Thursday 
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.40, 0.40, 0.40, 0.40, 0.40, 0.40, 0.40, 1.00, 1.00, 1.00, 1.00, 0.00,
    # Friday
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.40, 0.40, 0.40, 0.40, 0.40, 0.40, 0.40, 1.00, 1.00, 1.00, 1.00, 0.00,
    # Saturday
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.40, 0.40, 0.40, 0.40, 0.40, 0.40, 0.40, 1.00, 1.00, 1.00, 1.00, 0.00,
    # Sunday
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.40, 0.40, 0.40, 0.40, 0.40, 0.40, 0.40, 1.00, 1.00, 1.00, 1.00, 0.00
    ],
    "vent_profile": [
    # Monday
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Tuesday
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Wednesday event
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Thursday
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Friday event
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Saturday event
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Sunday event
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    ],
})
zone_definitions.append(("Zone 3", z3))

# --- ZONE 4: Offices & changing rooms ---------------------------------------------------------------------------------------------------------------------------------------
z4 = base_inputs.copy()
z4.update({
    "floor_area": 730.0,
    "area_roof": 295.0,
    "area_ground": 218.0,
    "room_volume": 2407.0,
    
    "rc_facade": 4.86,
    "thermal_mass_factor": 450000,

    "total_facade_areas": {'N': 0.0, 'NE': 13.0, 'E': 0.0, 'SE': 80.0, 'S': 0.0, 'SW': 150.0, 'W': 0.0, 'NW': 95.0},
    "window_percentages": {'N': 0.0, 'NE': 0.0, 'E': 0.0, 'SE': 0.475, 'S': 0.0, 'SW': 0.00, 'W': 0.00, 'NW': 0.642},
    "glazing_percentages": {'N': 0.00, 'NE': 0.00, 'E': 0.00, 'SE': 0.80, 'S': 0.00, 'SW': 0.00, 'W': 0.00, 'NW': 0.80},
    
    "max_people_per_m2": 0.06,  # based on 45 people
    "appliances_w_m2": 7.5,     # Average based on Vabi Elements: 10 for offices, 5 for changing rooms = 7.5
    "lighting_w_m2": 6.0,

    "heating_setpoint_profile": heating_profile_5,
    "cooling_setpoint_profile": cooling_profile_5,
        
    "floor_heating_activated": False,       # True or False, whether to include floor heating in the calculations
    "vent_flow_per_person": 25.2,
    "occ_profile": [
    # Monday workday
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 1.00, 1.00, 1.00, 1.00, 0.50, 1.00, 1.00, 1.00, 1.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
    # Tuesday workday
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 1.00, 1.00, 1.00, 1.00, 0.50, 1.00, 1.00, 1.00, 1.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
    # Wednesday workday
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 1.00, 1.00, 1.00, 1.00, 0.50, 1.00, 1.00, 1.00, 1.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
    # Thursday workday
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 1.00, 1.00, 1.00, 1.00, 0.50, 1.00, 1.00, 1.00, 1.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
    # Friday workday
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 1.00, 1.00, 1.00, 1.00, 0.50, 1.00, 1.00, 1.00, 1.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
    # Saturday weekend off
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
    # Sunday weekend off
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00
    ],
    "equip_profile": [
    # Monday workday
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 1.00, 1.00, 1.00, 1.00, 0.50, 1.00, 1.00, 1.00, 1.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
    # Tuesday workday
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 1.00, 1.00, 1.00, 1.00, 0.50, 1.00, 1.00, 1.00, 1.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
    # Wednesday workday
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 1.00, 1.00, 1.00, 1.00, 0.50, 1.00, 1.00, 1.00, 1.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
    # Thursday workday
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 1.00, 1.00, 1.00, 1.00, 0.50, 1.00, 1.00, 1.00, 1.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
    # Friday workday
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 1.00, 1.00, 1.00, 1.00, 0.50, 1.00, 1.00, 1.00, 1.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
    # Saturday weekend off
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
    # Sunday weekend off
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00
    ],
    "vent_profile": [
    # Monday
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Tuesday
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Wednesday event
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Thursday
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Friday event
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Saturday event
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Sunday event
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    ],
})
zone_definitions.append(("Zone 4", z4))

# --- ZONE 5: Workspace/workshop ---------------------------------------------------------------------------------------------------------------------------------------
z5 = base_inputs.copy()
z5.update({
    "floor_area": 480.0,
    "area_roof": 319.0,
    "area_ground": 160.0,
    "room_volume": 2495.0,

    "rc_facade": 4.86,
    "thermal_mass_factor": 450000,

    "total_facade_areas": {'N': 0.0, 'NE': 176.0, 'E': 0.0, 'SE': 110.0, 'S': 0.0, 'SW': 0.0, 'W': 0.0, 'NW': 74.0},
    "window_percentages": {'N': 0.0, 'NE': 0.034, 'E': 0.0, 'SE': 0.0, 'S': 0.0, 'SW': 0.0, 'W': 0.0, 'NW': 0.0},
    "glazing_percentages": {'N': 0.00, 'NE': 0.80, 'E': 0.00, 'SE': 0.00, 'S': 0.00, 'SW': 0.00, 'W': 0.00, 'NW': 0.00},
    
    "max_people_per_m2": 0.09,  # based on 44 people
    "appliances_w_m2": 5.0,
    "lighting_w_m2": 6.0,

    "heating_setpoint_profile": heating_profile_5,
    "cooling_setpoint_profile": cooling_profile_5,
    
    "floor_heating_activated": False,       # True or False, whether to include floor heating in the calculations
    "vent_flow_per_person": 25.2,
    "occ_profile": [
    # Monday
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 1.00, 1.00, 1.00, 1.00, 0.50, 1.00, 1.00, 1.00, 1.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
    # Tuesday 
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 1.00, 1.00, 1.00, 1.00, 0.50, 1.00, 1.00, 1.00, 1.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
    # Wednesday 
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 1.00, 1.00, 1.00, 1.00, 0.50, 1.00, 1.00, 1.00, 1.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
    # Thursday 
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 1.00, 1.00, 1.00, 1.00, 0.50, 1.00, 1.00, 1.00, 1.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
    # Friday
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 1.00, 1.00, 1.00, 1.00, 0.50, 1.00, 1.00, 1.00, 1.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
    # Saturday 
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
    # Sunday 
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00
    ],
    "equip_profile": [
    # Monday
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 1.00, 1.00, 1.00, 1.00, 0.50, 1.00, 1.00, 1.00, 1.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
    # Tuesday 
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 1.00, 1.00, 1.00, 1.00, 0.50, 1.00, 1.00, 1.00, 1.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
    # Wednesday 
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 1.00, 1.00, 1.00, 1.00, 0.50, 1.00, 1.00, 1.00, 1.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
    # Thursday 
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 1.00, 1.00, 1.00, 1.00, 0.50, 1.00, 1.00, 1.00, 1.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
    # Friday
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 1.00, 1.00, 1.00, 1.00, 0.50, 1.00, 1.00, 1.00, 1.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
    # Saturday 
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
    # Sunday 
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00
    ],
    "vent_profile": [
    # Monday
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Tuesday
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Wednesday event
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Thursday
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Friday event
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Saturday event
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Sunday event
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    ],
})
zone_definitions.append(("Zone 5", z5))

# --- ZONE 6/X: Storage/stairs ---------------------------------------------------------------------------------------------------------------------------------------
z6 = base_inputs.copy()
z6.update({
    "floor_area": 420.0,
    "area_roof": 360.0,
    "area_ground": 283.0,
    "room_volume": 3456.0,

    "rc_facade": 4.86,
    "thermal_mass_factor": 450000,

    "total_facade_areas": {'N': 0.0, 'NE': 130.0, 'E': 0.0, 'SE': 133.0, 'S': 0.0, 'SW': 53.0, 'W': 0.0, 'NW': 75.0},
    "window_percentages": {'N': 0.0, 'NE': 0.0, 'E': 0.0, 'SE': 0.083, 'S': 0.0, 'SW': 0.0, 'W': 0.0, 'NW': 0.0},
    "glazing_percentages": {'N': 0.00, 'NE': 0.00, 'E': 0.00, 'SE': 0.80, 'S': 0.00, 'SW': 0.00, 'W': 0.00, 'NW': 0.00},
    
    "max_people_per_m2": 0.03,  # based on 12 people
    "appliances_w_m2": 0.0,
    "lighting_w_m2": 0.0,
    
    "floor_heating_activated": False,       # True or False, whether to include floor heating in the calculations
    "vent_flow_per_person": 25.2,
    "heating_setpoint_profile": [16.0] * 168,
    "cooling_setpoint_profile": [28.0] * 168,
    "vent_cooling_setpoint": 27.0,
    "occ_profile": [
    # Monday 
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.10, 0.10,
    # Tuesday 
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.10, 0.10,
    # Wednesday event
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.30, 0.30, 0.30, 0.30, 0.30, 0.70, 0.70, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Thursday 
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.10, 0.10,
    # Friday event
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.30, 0.30, 0.30, 0.30, 0.30, 0.70, 0.70, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Saturday event
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.30, 0.30, 0.30, 0.30, 0.30, 0.70, 0.70, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Sunday event
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.30, 0.30, 0.30, 0.30, 0.30, 0.70, 0.70, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    ],
    "equip_profile": [
    # Monday 
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.10, 0.10,
    # Tuesday 
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.10, 0.10,
    # Wednesday event
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.30, 0.30, 0.30, 0.30, 0.30, 0.70, 0.70, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Thursday 
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.30, 0.10, 0.10,
    # Friday event
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.30, 0.30, 0.30, 0.30, 0.30, 0.70, 0.70, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Saturday event
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.30, 0.30, 0.30, 0.30, 0.30, 0.70, 0.70, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Sunday event
    0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.30, 0.30, 0.30, 0.30, 0.30, 0.70, 0.70, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    ],
    "vent_profile": [
    # Monday
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Tuesday
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Wednesday event
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Thursday
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Friday event
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Saturday event
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    # Sunday event
    0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00,
    ],
    "system_profile": [0.0]*168 # This zone has no emitters and is just a free floating zone
})
zone_definitions.append(("Zone 6", z6))

# =================================================================
# SECTION 4: MAIN SIMULATION LOOP (CASCADED LOGIC)
# =================================================================

# Upload the weather data including temperatures and solar radiation per orientation
file_path = 'Input Multi Zone Model - NEN5060-B2 1% solar orient - Extreme Year.csv'

try:
    weather_df = pd.read_csv(file_path, sep=';')
    weather_df.columns = weather_df.columns.str.strip()
    if 'temp_ext' in weather_df.columns and 'T' not in weather_df.columns:
        weather_df = weather_df.rename(columns={'temp_ext': 'T'})
    
    # Initialize all zone objects and store them in a dictionary
    zones = {}
    for name, params in zone_definitions:
        params['hourly_df'] = weather_df 
        # The __init__ call handles the internal setup
        zones[name] = ZoneModel(name, **params)

    # MASTER HOURLY LOOP (8760 Hours (length of weather dataset))
    for t in range(len(weather_df)):
        zone_external_flows = {name: 0.0 for name in zones.keys()}
        
        # Calculate Heat Exchange via Shared Walls
        for adj in adjacencies:
            z_a_name, z_b_name = adj["zones"]
            ua_value = adj["UA"]
            
            # Grabbing the current state from the class objects
            temp_a = zones[z_a_name].current_temp
            temp_b = zones[z_b_name].current_temp
            
            # Physics: Flow = U * A * Delta_T
            flow_b_to_a = ua_value * (temp_b - temp_a)
            
            # Storing the "push/pull" of heat for each zone
            zone_external_flows[z_a_name] += flow_b_to_a
            zone_external_flows[z_b_name] -= flow_b_to_a 

        # --- UFH Step 1: Determine Seasonal Mode for floor heating (Forecast Logic) ---
        forecast_end = min(t + 24, len(weather_df))
        upcoming_avg_temp = weather_df.iloc[t:forecast_end]['T'].mean()
        is_winter = True if upcoming_avg_temp < 15.5 else False

        # --- UFH Step 2: Weekly Schedule Check ---
        # Get hour of the week (0-167) to index the floor_profile
        hour_of_week = t % 168

        # --- UFH Step 3: Link Atrium Temperature cascadede air strategy---
        atrium_air_temp = zones['Zone 2'].current_temp

        # --- Update each zone's physics ---
        for name, zone_obj in zones.items():
            current_source = atrium_air_temp if name != 'Zone 2' else None
        
            # --- Specific logic for the Atrium (Zone 2) floor schedule ---
            if name == 'Zone 2':
                # Grab the configuration for Zone 2 (it's the second item in your list)
                z2_config = zone_definitions[1][1]
                
                # Check if heating is generally activated AND if the profile allows it this hour
                # Note: We use .get() just in case 'floor_profile' isn't in every zone
                schedule_val = z2_config.get('floor_profile', [1.0]*168)[hour_of_week]
                is_installed = z2_config.get('floor_heating_activated', False)
                
                # Update the object attribute immediately before the physics calculation
                zone_obj.floor_heating_activated = bool(is_installed and schedule_val > 0)
        
            # Physics calculation
            zone_obj.calculate_hour_step(t, external_q_flow=zone_external_flows[name], is_winter=is_winter, t_source=current_source)

    # =================================================================
    # SECTION 5: Save CSV of Hourly Results (Combined Heating + Cooling) for All Zones
    # =================================================================

    hourly_export_data = []
    for t in range(len(weather_df)):
        row = {"Hour": t + 1}
        for name, zone_obj in zones.items():
            if name == "Zone 6":
                continue
            # This net_demand now correctly includes the UFH energy
            net_demand = zone_obj.annual_heating_results[t] + zone_obj.annual_cooling_results[t]
            row[f"Demand_{name}"] = round(net_demand, 1)
        hourly_export_data.append(row)

    pd.DataFrame(hourly_export_data).to_csv("Input - Results_Yearly_Zone_Demands_Combined.csv", index=False, sep=',')
    
    # =================================================================
    # SECTION 6: Print statements and summary tables per zone
    # =================================================================

    # --- INITIALIZE RESULTS STORAGE ---
    final_results = []
    all_hourly_results = {"Hour": range(1, 8761)} 

    for name, zone_obj in zones.items():
        # Calculate Totals
        annual_h = sum(zone_obj.annual_heating_results)
        annual_c = sum(zone_obj.annual_cooling_results)

        # Collect the hourly demand (Heating + Cooling) for this specific zone
        all_hourly_results[f"Demand_{name}"] = [h + c for h, c in zip(zone_obj.annual_heating_results, zone_obj.annual_cooling_results)]
        
        # Calculate Peaks (using max/min from the hourly lists)
        peak_h = max(zone_obj.annual_heating_results) if zone_obj.annual_heating_results else 0.0
        peak_c = min(zone_obj.annual_cooling_results) if zone_obj.annual_cooling_results else 0.0
        
        # Print detailed results per zone (matching Multi_zone_model style)
        print(f"\n--- Results for {name} ---")
        print("Simulation Complete.")
        print(f"Room Volume: {zone_obj.room_volume:.2f} m3")
        print(f"Peak Heating: {peak_h:.2f} kW | Peak Cooling: {peak_c:.2f} kW")
        print(f"Total Annual Heat Demand: {annual_h:.2f} kWh")
        print(f"Total Annual Cool Demand: {annual_c:.2f} kWh")
        
        final_results.append({
            "Zone": name, 
            "Annual Heating [kWh]": round(annual_h, 2), 
            "Annual Cooling [kWh]": round(annual_c, 2)
        })

    # 3.5 TRACK PEAK TIMESTEPS
    hourly_results_df = pd.DataFrame(all_hourly_results)
    demand_columns = [col for col in hourly_results_df.columns if col.startswith('Demand_')]
    total_hourly_demand = hourly_results_df[demand_columns].sum(axis=1)

    peak_heating_val = total_hourly_demand.max()
    peak_heating_hour = total_hourly_demand.idxmax() + 1 

    peak_cooling_val = total_hourly_demand.min()
    peak_cooling_hour = total_hourly_demand.idxmin() + 1

    # =================================================================
    # SECTION 7: Final summary of building-wide performance
    # =================================================================

    print("\n" + "="*45)
    print("         MULTI-ZONE SIMULATION RESULTS         ")
    print("="*45)
  
    # Create the DataFrame
    results_df = pd.DataFrame(final_results)
    print(results_df)
    
    # Calculate Totals
    total_h = results_df["Annual Heating [kWh]"].sum()
    total_c = results_df["Annual Cooling [kWh]"].sum()
    
    print("-" * 45)
    print(f"TOTAL BUILDING HEATING: {total_h:,.2f} kWh")
    print(f"TOTAL BUILDING COOLING: {total_c:,.2f} kWh")
    print("="*45)

    print("-" * 45)
    print(f"BUILDING PEAK HEATING: {peak_heating_val:,.2f} kW at Hour {peak_heating_hour}")
    print(f"BUILDING PEAK COOLING: {peak_cooling_val:,.2f} kW at Hour {peak_cooling_hour}")
    print("-" * 45)
          
except Exception as e:
    print(f"An unexpected error occurred: {e}")

# =================================================================
# SECTION 8: Export hourly results to Excel for detailed analysis (One Zone style per sheet)
# =================================================================

export_filename = 'Building_Thermal_Performance_Detailed with setback with UFH.xlsx'

print(f"\nExporting detailed results to {export_filename}...")
print("Please wait, calculating column widths for all zones...")

# We use 'openpyxl' to match the logic from your One Zone Dynamic Model
with pd.ExcelWriter(export_filename, engine='openpyxl') as writer:
    # 1. Save the Global Summary Table first
    results_df.to_excel(writer, sheet_name='Annual_Summary', index=False)
    
    # Format Summary Sheet
    summary_ws = writer.sheets['Annual_Summary']
    for column_cells in summary_ws.columns:
        content_length = max(len(str(cell.value)) for cell in column_cells)
        adjusted_width = min(max(content_length, 10), 25)
        summary_ws.column_dimensions[column_cells[0].column_letter].width = adjusted_width
    
    # 2. Iterate through each zone object and export its 'One Zone' style data
    for name, zone_obj in zones.items():
        # Get the dataframe from the zone instance
        zone_detailed_df = zone_obj.get_hourly_dataframe()
        
        zone_detailed_df['Q_cool (kWh)'] = zone_obj.annual_cooling_results
        # Save to a new sheet named after the zone
        zone_detailed_df.to_excel(writer, sheet_name=name, index=False)
        
        # Access the openpyxl worksheet object for this specific zone
        worksheet = writer.sheets[name]
        
        # --- AUTO-ADJUST COLUMN WIDTHS (BASED ON YOUR SNIPPET) ---
        # Note: This checks every row (8760), so it may take a few seconds
        for column_cells in worksheet.columns:
            # Determine length based on cell content (header + data)
            content_length = max(len(str(cell.value)) for cell in column_cells)
            
            # Column width logic: min 10, max 25
            adjusted_width = min(max(content_length, 10), 25)
            
            # Apply the width to the column
            worksheet.column_dimensions[column_cells[0].column_letter].width = adjusted_width

print(f"Export Complete. Detailed results saved in: {export_filename}")

# =================================================================
# DYNAMIC GRID GENERATOR - kVA PROFILE (for TES Script)
# =================================================================

# 1. Configuration
TOTAL_GRID_LIMIT = 61.25    # The main breaker limit (kVA)
PUMP_POWER_W_M2  = 0.5     # Estimated W/m2 for circulation pumps

grid_kva_history = []

# Loop through every hour of the year
for t in range(8760):
    total_building_load_w = 0
    
    for name, zone_obj in zones.items():
        # Get schedules for this specific hour
        idx_168 = t % 168
        light_sch = zone_obj.equip_profile[idx_168]
        vent_sch  = zone_obj.vent_profile[idx_168]
        sys_active = zone_obj.system_profile[idx_168]
        
        # A. Lighting & Appliances
        z_light = zone_obj.lighting_w_m2 * zone_obj.floor_area * light_sch
        z_equip = zone_obj.appliances_w_m2 * zone_obj.floor_area * light_sch
        
        # B. Fans (Using RAW power, ignoring the 0.5 heat factor)
        z_fans = zone_obj.fans_power * vent_sch
        
        # C. Pumps (Active only when system is on)
        z_pumps = (PUMP_POWER_W_M2 * zone_obj.floor_area) if sys_active > 0 else 0
        
        total_building_load_w += (z_light + z_equip + z_fans + z_pumps)

    # Convert Watts to kVA (assuming Power Factor ~1.0)
    building_kva = total_building_load_w / 1000.0
    
    # Residual left for the Heat Pump
    available_for_hp = TOTAL_GRID_LIMIT - building_kva
    grid_kva_history.append(max(0, available_for_hp))

# 3. Export to CSV for TES Script
df_grid_final = pd.DataFrame({
    'Hour': range(1, 8761),
    'Grid_kVA': grid_kva_history
})

# Save using the filename your TES script expects
grid_filename = 'Input total kVA - Output dataset generator - Hour, Grid_kVA.csv'
df_grid_final.to_csv(grid_filename, index=False)

print(f"\n--- Dynamic Grid Profile Generated ---")
print(f"Max Available: {df_grid_final['Grid_kVA'].max():.2f} kVA (Night)")
print(f"Min Available: {df_grid_final['Grid_kVA'].min():.2f} kVA (Peak Occupancy)")

    


--- Results for Zone 1 ---
Simulation Complete.
Room Volume: 4077.00 m3
Peak Heating: 49.48 kW | Peak Cooling: -58.79 kW
Total Annual Heat Demand: 42989.05 kWh
Total Annual Cool Demand: -9223.77 kWh

--- Results for Zone 2 ---
Simulation Complete.
Room Volume: 7370.00 m3
Peak Heating: 46.52 kW | Peak Cooling: -17.05 kW
Total Annual Heat Demand: 28097.24 kWh
Total Annual Cool Demand: 11.85 kWh

--- Results for Zone 3 ---
Simulation Complete.
Room Volume: 4315.00 m3
Peak Heating: 68.68 kW | Peak Cooling: -55.07 kW
Total Annual Heat Demand: 34201.80 kWh
Total Annual Cool Demand: -17798.05 kWh

--- Results for Zone 4 ---
Simulation Complete.
Room Volume: 2407.00 m3
Peak Heating: 65.30 kW | Peak Cooling: -62.62 kW
Total Annual Heat Demand: 27962.75 kWh
Total Annual Cool Demand: -8435.07 kWh

--- Results for Zone 5 ---
Simulation Complete.
Room Volume: 2495.00 m3
Peak Heating: 44.98 kW | Peak Cooling: -39.77 kW
Total Annual Heat Demand: 23348.01 kWh
Total Annual Cool Demand: -4061.81 kWh

-